<a href="https://colab.research.google.com/github/Berkaycetinn/log-analyzer/blob/main/03_embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
!pip install -q drain3 sentence-transformers
!git clone --depth 1 https://github.com/logpai/loghub.git

import re
import numpy as np
import pandas as pd
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
from drain3.masking import MaskingInstruction

fatal: destination path 'loghub' already exists and is not an empty directory.


In [40]:
pattern = re.compile(
    r"^(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2},\d{3}) "
    r"(?P<level>[A-Z]+) \[(?P<thread>[^\]]*)\] (?P<component>\S+): (?P<message>.*)$"
)
rows = []
with open("loghub/Hadoop/Hadoop_2k.log", encoding="utf-8") as f:
    for line in f:
        m = pattern.match(line.rstrip("\n"))
        if m:
            rows.append(m.groupdict())
df = pd.DataFrame(rows)

In [41]:
config = TemplateMinerConfig()
config.drain_sim_th = 0.4
config.drain_depth = 4
config.masking_instructions = [
    MaskingInstruction(r"\b(?:application|appattempt|container|job|task|attempt)_[\w]+", "ID"),
    MaskingInstruction(r"blk_-?\d+(?:_\d+)?", "BLK"),
    MaskingInstruction(r"\d+\.\d+\.\d+\.\d+", "IP"),
    MaskingInstruction(r"-?\d+(?:ms|s)\b", "DUR"),
    MaskingInstruction(r"\b\d+\b", "NUM"),
]
miner = TemplateMiner(config=config)
df["cluster_id"] = [miner.add_log_message(m)["cluster_id"] for m in df["message"]]
tmap = {c.cluster_id: c.get_template() for c in miner.drain.clusters}
df["template"] = df["cluster_id"].map(tmap)

In [42]:
SUSP = re.compile(r"fail|exception|error|timeout|timed out|refused|no route|unreachable|killed|denied|retry|retrying", re.I)
BENIGN = re.compile(r"maxTaskFailuresPerNode")
df["suspicious"] = df["message"].apply(lambda m: bool(SUSP.search(m)) and not BENIGN.search(m))
df["important"] = df["level"].isin(["ERROR", "WARN", "FATAL"]) | df["suspicious"]

In [43]:
# --- 4. Template tablosu ---
tpl = (df.groupby("cluster_id")
         .agg(template=("template", "first"),
              component=("component", lambda s: s.mode()[0]),
              count=("message", "size"),
              n_important=("important", "sum"),
              first=("time", "min"), last=("time", "max"))
         .reset_index())
tpl["imp_frac"] = tpl["n_important"] / tpl["count"]
tpl["important"] = tpl["imp_frac"] >= 0.5
tpl["text_raw"] = tpl["template"]

print("Satır:", len(df), "| Template:", len(tpl))

Satır: 2000 | Template: 99


In [44]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Cihaz:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

def embed(model_name, texts):
    model = SentenceTransformer(model_name, device=device)
    return model.encode(list(texts), normalize_embeddings=True, show_progress_bar=False)

Cihaz: cuda NVIDIA RTX PRO 6000 Blackwell Server Edition


In [45]:
emb_final = embed("all-mpnet-base-v2", tpl["text_raw"])
np.save("embeddings.npy", emb_final)
tpl.to_csv("templates.csv", index=False)

print("Vektör şekli:", emb_final.shape)   # (99, 768)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vektör şekli: (99, 768)


In [46]:
def show_neighbors(emb, key, k=5):
    i = tpl.index[tpl["template"].str.contains(key, regex=False)][0]
    sim = emb @ emb[i]
    order = np.argsort(-sim)[:k + 1]
    print(f"\n=== {key} ===")
    for j in order:
        mark = "ÖNEMLİ" if tpl.loc[j, "important"] else "      "
        print(f"{sim[j]:.2f} {mark} ({tpl.loc[j,'count']:3d}x) {tpl.loc[j,'template'][:95]}")

for key in ["NoRouteToHostException: No Route", "Failed to renew lease",
            "Container complete event for unknown", "Address change detected"]:
    show_neighbors(emb, key)


=== NoRouteToHostException: No Route ===
1.00 ÖNEMLİ (  2x) Task: <ID> - exited : java.net.NoRouteToHostException: No Route to Host from MININT-FNANLI5/<IP
0.96 ÖNEMLİ (  4x) Diagnostics report from <ID>: Error: java.net.NoRouteToHostException: No Route to Host from MIN
0.54 ÖNEMLİ (  1x) Error writing History Event: org.apache.hadoop.mapreduce.jobhistory.TaskAttemptUnsuccessfulComp
0.49        (  1x) Extract jar:file:/D:/hadoop-<NUM>.<NUM>.<NUM>-localbox/share/hadoop/yarn/hadoop-yarn-common-<NU
0.48        (  1x) Event Writer setup for JobId: <ID>, File: hdfs://msra-sa-<NUM>:<NUM>/tmp/hadoop-yarn/staging/ms
0.48        (  1x) Adding protocol org.apache.hadoop.mapreduce.v2.api.MRClientProtocolPB to the server

=== Failed to renew lease ===
1.00 ÖNEMLİ (326x) Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for <NUM> seconds. Will retry s
0.43 ÖNEMLİ (146x) Retrying connect to server: msra-sa-<NUM>:<NUM>. Already tried <NUM> time(s); retry policy is R
0.43 ÖNEMLİ (  1x) 

In [47]:
r = df[df["message"].str.contains("Retrying connect", regex=False)]
print("Retrying connect:", len(r), "satır |", r["time"].min(), "->", r["time"].max(), "|", r["level"].unique())

keys = ["Address change detected", "Failed to renew lease", "ERROR IN CONTACTING RM",
        "Retrying connect", "NoRouteToHostException: No Route"]
sel = tpl[tpl["template"].apply(lambda t: any(k in t for k in keys))].copy()
sel["template"] = sel["template"].str[:55]
print(sel[["template", "count", "first", "last"]].to_string(index=False))

Retrying connect: 146 satır | 18:06:03,856 -> 18:10:54,546 | ['INFO']
                                               template  count        first         last
Address change detected. Old: msra-sa-<NUM>/<IP>:<NUM>     476 18:05:27,570 18:10:55,202
Failed to renew lease for [DFSClient_NONMAPREDUCE_15378    326 18:05:27,570 18:10:54,202
                                ERROR IN CONTACTING RM.    147 18:06:01,840 18:10:54,546
Retrying connect to server: msra-sa-<NUM>:<NUM>. Alread    146 18:06:03,856 18:10:54,546
Task: <ID> - exited : java.net.NoRouteToHostException:       2 18:06:26,029 18:06:28,217
Diagnostics report from <ID>: Error: java.net.NoRouteTo      4 18:06:26,029 18:06:28,217


In [48]:
gt = pd.read_csv("loghub/Hadoop/Hadoop_2k.log_structured.csv")

a = df["message"].str.strip().str.split().str.join(" ")
b = gt["Content"].astype(str).str.strip().str.split().str.join(" ")
print("Boşluklar normalize edilince eşleşme:", (a.values == b.values).mean())

bad = df.index[a.values != b.values]
print("Eşleşmeyen satır:", len(bad))
for i in bad[:5]:
    print(i, "\n  bizim:", repr(df.loc[i, "message"])[:150],
             "\n  gt   :", repr(gt.loc[i, "Content"])[:150])

Boşluklar normalize edilince eşleşme: 1.0
Eşleşmeyen satır: 0


In [49]:
def important_neighbor_rate(emb, k=3):
    sim = emb @ emb.T
    np.fill_diagonal(sim, -1)
    nn = np.argsort(-sim, axis=1)[:, :k]
    lab = tpl["important"].values
    imp = np.where(lab)[0]
    return lab[nn[imp]].mean()

WORDS = {"<*>": " ", "<ID>": " id ", "<NUM>": " number ", "<IP>": " ip address ",
         "<DUR>": " duration ", "<BLK>": " block "}

def clean(t):
    for k, v in WORDS.items():
        t = t.replace(k, v)
    t = re.sub(r"([a-z])([A-Z])", r"\1 \2", t)
    t = re.sub(r"(?<=\w)[._$](?=\w)", " ", t)
    return re.sub(r"\s+", " ", t).strip()

tpl["text_clean"] = tpl["template"].apply(clean)

print(f"Taban oran (rastgele): {tpl['important'].mean():.2f}\n")
for mname in ["all-MiniLM-L6-v2", "all-mpnet-base-v2", "BAAI/bge-base-en-v1.5"]:
    for col in ["text_raw", "text_clean"]:
        e = embed(mname, tpl[col])
        print(f"{mname:26s} {col:11s} -> {important_neighbor_rate(e):.2f}")

Taban oran (rastgele): 0.17



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

all-MiniLM-L6-v2           text_raw    -> 0.49


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

all-MiniLM-L6-v2           text_clean  -> 0.47


...

all-mpnet-base-v2          text_clean  -> 0.41


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BAAI/bge-base-en-v1.5      text_raw    -> 0.43


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BAAI/bge-base-en-v1.5      text_clean  -> 0.45


In [50]:
print("Örneğin son satırı:", df["time"].max())

lease = df[df["message"].str.contains("Failed to renew lease", regex=False)]
print(lease.iloc[0]["time"], "|", lease.iloc[0]["message"])
print(lease.iloc[-1]["time"], "|", lease.iloc[-1]["message"])

Örneğin son satırı: 18:10:55,202
18:05:27,570 | Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for 30 seconds.  Will retry shortly ...
18:10:54,202 | Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for 356 seconds.  Will retry shortly ...


In [51]:
# 1) Önemli satır / template sayısı
print("Önemli satır:", int(df["important"].sum()), "| Önemli template:", int(tpl["important"].sum()))

# 2) Vektör şekli
print("Vektör şekli:", emb_final.shape)

# 3) Retrying connect komşuları
show_neighbors(emb_final, "Retrying connect")

# 4) Örnek nerede bitiyor, lease sayacı nereden başlıyor
print("\nÖrneğin son satırı:", df["time"].max())
lease = df[df["message"].str.contains("Failed to renew lease", regex=False)]
print(lease.iloc[0]["time"], "|", lease.iloc[0]["message"])
print(lease.iloc[-1]["time"], "|", lease.iloc[-1]["message"])

Önemli satır: 1121 | Önemli template: 17
Vektör şekli: (99, 768)

=== Retrying connect ===
1.00 ÖNEMLİ (146x) Retrying connect to server: msra-sa-<NUM>:<NUM>. Already tried <NUM> time(s); retry policy is R
0.56        (  1x) Connecting to ResourceManager at msra-sa-<NUM>/<IP>:<NUM>
0.41 ÖNEMLİ (  4x) Diagnostics report from <ID>: Error: java.net.NoRouteToHostException: No Route to Host from MIN
0.39 ÖNEMLİ (  2x) Task: <ID> - exited : java.net.NoRouteToHostException: No Route to Host from MININT-FNANLI5/<IP
0.38        (  1x) Instantiated MRClientService at MININT-FNANLI5.fareast.corp.microsoft.com/<IP>:<NUM>
0.37 ÖNEMLİ (476x) Address change detected. Old: msra-sa-<NUM>/<IP>:<NUM> New: msra-sa-<NUM>:<NUM>

Örneğin son satırı: 18:10:55,202
18:05:27,570 | Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for 30 seconds.  Will retry shortly ...
18:10:54,202 | Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for 356 seconds.  Will retry shortly ...
